# Lesson 2b: Writing Workers (Guppy)

```{important}
This is an alternative lesson to [Lesson 2a](pytket_graph.ipynb).
It uses [Guppy](https://guppylang.org) to construct the quantum computation.
```

In Lesson 2a you wrote a worker that constructs a symbolic quantum circuit and running it on a simulator.
In this example we're going to extend your "my_example_worker" with Guppy functions.
1. Define a simple parametric Guppy program
2. Provide parameters at runtime

```{note}
Guppy is a programming language for Quantum Computers.
All code in a Guppy program will be executed in coherence time on the controller.
Since it is embedded in Python, we can use the same worker mechanism as before.

```
[Lesson 3](./storage_and_executors.ipynb) will proceed with the pytket functionality, so you may skip ahead.

## Prerequisite

We're going to extend the existing worker so we assume you have run the prerequisites from before.

As a reminder this was (see [2a](pytket_graph.ipynb#Prerequisite)):

```bash
uv init
uv add tierkreis pytket pytket-qiskit sympy ruff
uv run tkr project init
uv run tkr init worker -n my_example_worker
```
We're adding the dependencies for Guppy:
```bash
uv add guppylang hugr
```

## Defining the tasks

As before we assume your implementing `impl.py` of your previously defined worker

In [1]:
# The worker is added here for validity of the example
from tierkreis import Worker

worker = Worker("my_example_worker")
# Your previous tasks will be here

In Guppy, you write programs by using the `@guppy` annotation and then calling compile.
You can do the same from inside a worker function, here we will dynamically construct a n-qubit GHZ state.

In [2]:
from guppylang import comptime, guppy
from guppylang.std.builtins import array, result
from guppylang.std.quantum import cx, h, measure_array, qubit
from hugr.package import Package


@worker.task()
def ghz(size: int) -> Package:
    n = guppy.nat_var("n")

    @guppy
    def build_ghz_state(q: array[qubit, n]) -> None:  # type: ignore
        h(q[0])
        for i in range(n - 1):  # type: ignore
            cx(q[i], q[i + 1])

    @guppy
    def main() -> None:
        q = array(qubit() for _ in range(comptime(size)))  # type: ignore
        build_ghz_state(q)

        result("c", measure_array(q))

    return main.compile()

As before you can provide inputs to the function.
```{important}
You must call `compile()` on the main you wish to run as a task.
This will generate a serialized version of the program (a Hugr), which can be used as value in Tierkreis. The corresponding type is `hugr.Package`.
```


## Generating stubs

As before generate the api from the cli
```bash
uv run tkr init stubs
```

## Using the tasks

Now you can use the newly declared tasks in a graph similar to how you used the `builtin` functionality or other tasks.
You have to import the task API from the worker first which you then can use with a task node.
First we declare the graph

In [3]:
# Constructing, put into graphs/main.py
from tierkreis.builder import Graph
from tierkreis.controller.data.models import TKR

graph = Graph(TKR[int], TKR[Package])

and then add the tasks:

In [4]:
# Constructing, put into graphs/main.py
from my_example_worker import ghz  # noqa: F811

program = graph.task(ghz(graph.inputs))
workflow = graph.finish_with_outputs(program)  # type: ignore

## Running the graph
As before you know can run the graph, the circuit we have defined already above. The result is an binary represantion of the program.

In [5]:
# Running, put into graphs/main.py
from uuid import UUID

from tierkreis.controller import run_graph
from tierkreis.executor import ShellExecutor
from tierkreis.storage import FileStorage, read_outputs

storage = FileStorage(workflow_id=UUID(int=12347), name="Guppy example graph")
storage.clean_graph_files()
executor = ShellExecutor(registry_path=None, workflow_dir=storage.workflow_dir)
run_graph(storage, executor, workflow, 3)  # single import
output = read_outputs(workflow, storage)
print(output)

TierkreisError: Graph encountered errors